# UNI 03 — inference (submission)



In [ ]:
import os, numpy as np, pandas as pd
import torch, torch.nn as nn, timm
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
try:
    import openslide; HAVE_OPENSLIDE = True
except Exception:
    HAVE_OPENSLIDE = False
print(device, '| openslide', HAVE_OPENSLIDE)

In [ ]:
data_dir    = '/kaggle/input/competitions/prostate-cancer-grade-assessment'
UNI_WEIGHTS = '/kaggle/input/datasets/shashaboii/panda-uni-embeddings1/uni_weights/uni_pytorch_model.bin'   # from UNI 01 output
HEAD_PATH   = '/kaggle/input/datasets/shashaboii/panda-uni-head/uni_mil_fold0.pth'                # from UNI 02 output
LEVEL = 1; TILE = 256; N_TILES = 36; TILE_BATCH = 32; EMB_DIM = 1024; N_CLASSES = 6

df_test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
df_train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_dir = os.path.join(data_dir, 'test_images')
is_test = os.path.exists(test_dir)
folder = test_dir if is_test else os.path.join(data_dir, 'train_images')
df = df_test if is_test else df_train.loc[:50].copy()
print('is_test:', is_test, '| slides:', len(df))

In [ ]:
def read_slide(path, level=LEVEL):
    if HAVE_OPENSLIDE:
        s = openslide.OpenSlide(path); lv = min(level, s.level_count - 1)
        img = s.read_region((0, 0), lv, s.level_dimensions[lv]).convert('RGB'); s.close()
        return np.asarray(img)
    import tifffile
    with tifffile.TiffFile(path) as tif:
        ser = tif.series[0]
        arr = ser.levels[min(level, len(ser.levels)-1)].asarray() if len(ser.levels) > 1 else ser.asarray()
    arr = np.asarray(arr)
    if arr.ndim == 2: arr = np.stack([arr]*3, -1)
    return arr[..., :3]

def get_tiles(img, mode=0):
    h, w, _ = img.shape
    pad_h = (TILE-h % TILE) % TILE + ((TILE*mode)//2)
    pad_w = (TILE-w % TILE) % TILE + ((TILE*mode)//2)
    img2 = np.pad(img, [[pad_h//2, pad_h-pad_h//2],[pad_w//2, pad_w-pad_w//2],[0,0]], constant_values=255)
    img3 = img2.reshape(img2.shape[0]//TILE, TILE, img2.shape[1]//TILE, TILE, 3)
    img3 = img3.transpose(0,2,1,3,4).reshape(-1, TILE, TILE, 3)
    if len(img3) < N_TILES:
        img3 = np.pad(img3, [[0, N_TILES-len(img3)],[0,0],[0,0],[0,0]], constant_values=255)
    idxs = np.argsort(img3.reshape(img3.shape[0], -1).sum(-1))[:N_TILES]
    return img3[idxs]                                   # (N_TILES, TILE, TILE, 3) RGB, NOT inverted

import torch.nn.functional as F
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def tiles_to_uni(tiles):                                # UNI expects normal H&E, ImageNet norm, 224px
    x = torch.from_numpy(tiles).float().permute(0, 3, 1, 2) / 255.0
    x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
    return (x - _MEAN) / _STD

In [ ]:
def build_uni():
    return timm.create_model('vit_large_patch16_224', img_size=224, patch_size=16,
                             init_values=1e-5, num_classes=0, dynamic_img_size=True)

def load_uni(local_weights=None):
    m = build_uni()
    if local_weights and os.path.exists(local_weights):           # offline (Kaggle submission)
        sd = torch.load(local_weights, map_location='cpu')
    else:                                                         # online (download once)
        from huggingface_hub import hf_hub_download
        sd = torch.load(hf_hub_download('MahmoodLab/UNI', filename='pytorch_model.bin'),
                        map_location='cpu')
    m.load_state_dict(sd, strict=True)
    for p in m.parameters(): p.requires_grad_(False)
    return m.eval()
uni = load_uni(local_weights=UNI_WEIGHTS).to(device)
print('UNI loaded from local weights (offline)')

In [ ]:
@torch.no_grad()
def embed_slide(image_id, folder):
    tiles = get_tiles(read_slide(os.path.join(folder, f'{image_id}.tiff')), 0)
    x = tiles_to_uni(tiles).to(device)
    out = []
    for i in range(0, len(x), TILE_BATCH):
        with torch.autocast('cuda', dtype=torch.float16):
            out.append(uni(x[i:i+TILE_BATCH]).float().cpu())
    return torch.cat(out).numpy().astype(np.float16)              # (N_TILES, EMB_DIM)

In [ ]:
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_dim=EMB_DIM, hid=256, att=128, out_dim=5, p=0.25):
        super().__init__()
        self.fc   = nn.Sequential(nn.Linear(in_dim, hid), nn.ReLU(), nn.Dropout(p))
        self.attV = nn.Linear(hid, att); self.attU = nn.Linear(hid, att); self.attw = nn.Linear(att, 1)
        self.head = nn.Linear(hid, out_dim)
    def forward(self, x):                                          # x: (B, N, D)
        h = self.fc(x)
        a = torch.tanh(self.attV(h)) * torch.sigmoid(self.attU(h))
        a = torch.softmax(self.attw(a).squeeze(-1), dim=1)
        m = torch.bmm(a.unsqueeze(1), h).squeeze(1)
        return self.head(m), a
head = GatedAttentionMIL().to(device)
head.load_state_dict(torch.load(HEAD_PATH, map_location='cpu')); head.eval()
print('head loaded')

In [ ]:
@torch.no_grad()
def predict_slide(image_id):
    emb = torch.from_numpy(embed_slide(image_id, folder).astype(np.float32)).unsqueeze(0).to(device)
    logits, _ = head(emb)
    return float(torch.sigmoid(logits).sum(1).clamp(0, N_CLASSES-1).item())

scores = [predict_slide(i) for i in tqdm(df.image_id.tolist(), desc='predict')]
df['isup_grade'] = np.round(scores).astype(int)
df[['image_id', 'isup_grade']].to_csv('submission.csv', index=False)
print(df[['image_id', 'isup_grade']].head())

In [ ]:
pd.Series(df.isup_grade).value_counts().sort_index().plot.bar(color='#534AB7')
plt.title('Predicted ISUP distribution'); plt.xlabel('ISUP'); plt.ylabel('# slides'); plt.show()
